# MoReS Training & Evaluation
**Bi et al., "LLaVA Steering", ACL 2025 | L4 24GB | Qwen2.5-VL-3B bf16**

## Step 1: Install + Mount Drive

In [ ]:
!pip install -q torch torchvision transformers>=4.45 peft>=0.7 accelerate>=0.21 datasets>=2.14
!pip install -q bitsandbytes pyyaml pillow

from google.colab import drive
drive.mount('/content/drive')

import sys, os
sys.path.insert(0, '/content/drive/MyDrive/mmit/src')

import mmit
print('mmit loaded!')
print('Methods:', mmit.registry.list('training_method'))

## Step 2: Load Model (bf16)

In [ ]:
import torch
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM

MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"

print(f'Loading {MODEL} in bf16...')
processor = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
model = AutoVLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True,
)
print(f'Loaded! GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB / 24 GB')

## Step 3: Prepare MoReS + Load Data

In [ ]:
from mmit.training.methods.mores import MoReSMethod
from mmit.data.adapters.hf_datasets import HFDatasetsAdapter

# ═══ MoReS config (paper optimal) ═══
mores_config = {
    "intervention_rank": 1,
    "positions": "f4+l5",
    "dropout": 0.05,
    "share_weights": True,
    "steer_visual_only": False,
    "steer_ratio": 1.0,
}

method = MoReSMethod()
prepared_model, info = method.prepare_model(model, processor, mores_config)
print(info)
print()

# ═══ Load dataset (Paper Stage 3: task-specific, full) ═══
DATASET     = "lmms-lab/ScienceQA"
SPLIT       = "train"
MAX_SAMPLES = 0    # 0 = ALL (full dataset, ~12K)

print(f'Loading {DATASET} ({SPLIT}, {"ALL" if MAX_SAMPLES == 0 else MAX_SAMPLES})...')
adapter = HFDatasetsAdapter(
    dataset_name=DATASET,
    split=SPLIT,
    max_samples=MAX_SAMPLES if MAX_SAMPLES > 0 else None,
    streaming=False,
    load_images=True,
)
samples = list(adapter)
print(f'Loaded {len(samples)} samples')

## Step 4: Train

In [ ]:
import time, math, os, traceback
from torch.optim import AdamW
from torch.utils.data import DataLoader

# ═══ Training config (Paper Stage 3) ═══
EPOCHS     = 10          # paper: 10 epochs per task
LR         = 2e-4        # paper: 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 32          # effective batch size = 1 * 32 = 32 (paper: 32)
SAVE_EVERY = 2           # save checkpoint every N epochs
OUTPUT_DIR = "/content/drive/MyDrive/mmit_output/mores_scienceqa"
# ═══════════════════════════════════════

os.makedirs(OUTPUT_DIR, exist_ok=True)

def collate(batch_samples):
    from mmit.data.adapters.hf_datasets import decode_sample_image
    all_texts, all_images = [], []
    for s in batch_samples:
        img = decode_sample_image(s)
        q = s.question if hasattr(s, 'question') else str(s.turns[0].content if s.turns else '')
        a = s.ground_truth if hasattr(s, 'ground_truth') else str(s.turns[-1].content if len(s.turns) > 1 else '')
        messages = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]},
            {"role": "assistant", "content": [{"type": "text", "text": a}]},
        ]
        text = processor.apply_chat_template(messages, tokenize=False)
        all_texts.append(text)
        all_images.append(img)

    inputs = processor(
        text=all_texts, images=all_images,
        return_tensors="pt", padding=True, truncation=True,
    )
    inputs["labels"] = inputs["input_ids"].clone()
    return inputs

loader = DataLoader(samples, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)

param_groups = method.get_trainable_params(prepared_model)
optimizer = AdamW(param_groups, lr=LR, weight_decay=0.0)
total_steps = len(loader) * EPOCHS // GRAD_ACCUM

print(f'Dataset:  {len(samples)} samples')
print(f'Training: {EPOCHS} epochs, {len(loader)} steps/epoch, {total_steps} optimizer steps')
print(f'Batch:    {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} effective')
print(f'Params:   {sum(p.numel() for pg in param_groups for p in pg["params"]):,}')
print(f'GPU:      {torch.cuda.memory_allocated()/1024**3:.1f} GB')
print('=' * 60)

prepared_model.train()
global_step = 0
running_loss = 0.0
t0 = time.time()
error_hit = False

for epoch in range(EPOCHS):
    if error_hit:
        break
    epoch_loss = 0.0
    epoch_steps = 0

    for step, batch in enumerate(loader):
        device = next(prepared_model.parameters()).device
        moved = {}
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                v = v.to(device)
                if k == "pixel_values":
                    v = v.to(torch.bfloat16)
                moved[k] = v
            else:
                moved[k] = v

        moved["labels"] = method.preprocess_labels(
            moved["input_ids"], moved["labels"]
        )

        try:
            outputs = prepared_model(**{k: v for k, v in moved.items() if k != "image_tensor"})
            loss, metrics = method.compute_loss(prepared_model, moved, outputs)
            loss = loss / GRAD_ACCUM
            loss.backward()
        except Exception as e:
            print(f'\n  [ERROR] Epoch {epoch+1} Step {step}:')
            traceback.print_exc()
            optimizer.zero_grad()
            error_hit = True
            break

        epoch_loss += loss.item() * GRAD_ACCUM
        epoch_steps += 1
        running_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for pg in param_groups for p in pg["params"]], 1.0
            )
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % 20 == 0 or global_step == 1:
                avg_loss = running_loss / (global_step * GRAD_ACCUM + epoch_steps % GRAD_ACCUM)
                elapsed = time.time() - t0
                eta = elapsed / global_step * (total_steps - global_step)
                m, s = divmod(int(eta), 60)
                h, m = divmod(m, 60)
                eta_str = f'{h}h{m:02d}m' if h else f'{m}m{s:02d}s'
                print(f'  E{epoch+1}/{EPOCHS} Step {global_step}/{total_steps} | '
                      f'Loss: {loss.item()*GRAD_ACCUM:.4f} | '
                      f'Avg: {epoch_loss/epoch_steps:.4f} | '
                      f'ETA: {eta_str}')

    if not error_hit:
        avg_ep = epoch_loss / max(epoch_steps, 1)
        print(f'  --- Epoch {epoch+1}/{EPOCHS} done | Avg loss: {avg_ep:.4f} ---')

        # Save checkpoint every N epochs
        if (epoch + 1) % SAVE_EVERY == 0 or (epoch + 1) == EPOCHS:
            ckpt_path = os.path.join(OUTPUT_DIR, f'epoch_{epoch+1}')
            os.makedirs(ckpt_path, exist_ok=True)
            metadata = {"model_id": MODEL, "config": mores_config, "epoch": epoch+1}
            method.save_checkpoint(prepared_model, processor, ckpt_path, metadata)
            print(f'  Checkpoint saved: {ckpt_path}')

if not error_hit:
    # Save final checkpoint at top level too
    metadata = {"model_id": MODEL, "config": mores_config, "epoch": EPOCHS}
    method.save_checkpoint(prepared_model, processor, OUTPUT_DIR, metadata)
    elapsed = time.time() - t0
    m, s = divmod(int(elapsed), 60)
    h, m = divmod(m, 60)
    print(f'\nTraining complete! {h}h{m:02d}m{s:02d}s total')
    print(f'Final checkpoint: {OUTPUT_DIR}')
else:
    print('\nTraining stopped due to error.')

## Step 5: Quick Eval (1 sample)

In [ ]:
from datasets import load_dataset
from PIL import Image
from IPython.display import display

# Eval on ScienceQA validation
ds = load_dataset("lmms-lab/ScienceQA", split="validation", streaming=True)
sample = next(iter(ds))

image = sample.get('image')
if image is not None:
    image = image.convert('RGB')
else:
    from PIL import Image as PILImage
    image = PILImage.new('RGB', (224, 224), (128, 128, 128))

question = sample.get('question', '')
choices = sample.get('choices', [])
answer_idx = sample.get('answer', 0)
gt = choices[answer_idx] if choices and answer_idx < len(choices) else str(answer_idx)

if choices:
    options = '\n'.join(f'({chr(65+i)}) {c}' for i, c in enumerate(choices))
    full_question = f'{question}\n{options}\nAnswer with the letter.'
else:
    full_question = question

display(image.resize((300, 300)))
print(f'Question:     {question}')
if choices:
    print(f'Choices:      {choices}')
print(f'Ground Truth: ({chr(65+answer_idx)}) {gt}')
print('-' * 40)

prepared_model.eval()
device = next(prepared_model.parameters()).device
messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': full_question}]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True)

moved = {}
for k, v in inputs.items():
    if isinstance(v, torch.Tensor):
        v = v.to(device)
        if k == 'pixel_values': v = v.to(torch.bfloat16)
        moved[k] = v
    else: moved[k] = v

with torch.no_grad():
    out = prepared_model.generate(**moved, max_new_tokens=64, do_sample=False)
pred = processor.decode(out[0][moved['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print(f'Prediction:   {pred}')
print(f'Match:        {"YES" if gt.lower() in pred.lower() or chr(65+answer_idx) in pred else "NO"}')